# Trabajo Práctico: Consumo de API con Python

**Alumno:** Claudio Segura Blanc, Ezequiel Enrico Areco, Pablo Benitez, Natalia Mascardini,    
**Curso:** ARQWEB Polo Marcos Paz   
**Materia:**  Programacion Web II   
**Fecha:**  27/3/2026   

## Objetivo
Consumir una API pública utilizando distintos métodos HTTP y mostrar los resultados en DataFrames.  
# Primer paso crear proyecto para el tp2  

mkdir tp2  
cd tp2  
# Generamos entorno  
python3 -m venv .venv  
. .venv/bin/activate  
pip install requests  
pip install Flask  
pip install pandas  
pip install jupyterlab  
# Ejecucion de editor jupyter para el presente documento.
jupyter lab  



# Para el proyecto elegi dos opciones para probar todos los metodos:  
restful-booker.herokuapp.com posee una api para poder reservas de hotel. 


# Preliminar armar Token que necesita la web (api) de ejemplo:

In [10]:
import requests
import pandas as pd

url = "https://restful-booker.herokuapp.com"
print("API Restful Booker para probar tp2")

# Crear token la api necesita un token para autenticar las peticiones, se obtiene con el endpoint /auth
auth_data = {
    "username": "admin",
    "password": "password123"
}

respuesta = requests.post(url + "/auth", json=auth_data)
print("Status Code:", respuesta.status_code)

token = None
if respuesta.status_code == 200:
    token = respuesta.json()["token"]
    print("Token creado:", token)
else:
    print("Error al crear token")
    print(respuesta.text)

API Restful Booker para probar tp2
Status Code: 200
Token creado: e05f5a39ed807ac


# Metodo GET ejemplo:

In [11]:
respuesta = requests.get(url + "/booking")
print("GET /booking - Status:", respuesta.status_code)

if respuesta.status_code == 200:
    data = respuesta.json()
    df = pd.DataFrame(data)
    display(df.head(10))   # Muestra las primeras 10 filas
else:
    print(respuesta.text)

GET /booking - Status: 200


,bookingid
0,1
1,2
2,3
3,4
4,5
5,6
6,7
7,8
8,9
9,10


# Metodo Get para un Id en particular:

In [12]:
# Tomamos el primer bookingid de la lista anterior
if 'df' in locals() and not df.empty:
    booking_id = df.iloc[0]['bookingid']
    print("Probando con booking_id:", booking_id)
    
    respuesta = requests.get(url + f"/booking/{booking_id}")
    print("Status Code:", respuesta.status_code)
    
    if respuesta.status_code == 200:
        datos = respuesta.json()
        df_booking = pd.DataFrame([datos])   # Convertimos el diccionario a DataFrame
        display(df_booking)
    else:
        print(respuesta.text)

Probando con booking_id: 1
Status Code: 200


,firstname,lastname,totalprice,depositpaid,bookingdates
0,Eric,Wilson,719,False,"{'checkin': '2021-03-21', 'checkout': '2021-09..."


# Método POST ejemplo:

In [13]:
#armamos una reserva para probar!!!
reserva_booking = {
    "firstname": "Claudio",
    "lastname": "Segura",
    "totalprice": 15000,
    "depositpaid": True,
    "bookingdates": {
        "checkin": "2026-04-15",
        "checkout": "2026-04-18"
    },
    "additionalneeds": "Desayuno incluido"
}

respuesta = requests.post(url + "/booking", json=reserva_booking)
print("POST /booking - Status:", respuesta.status_code)

if respuesta.status_code == 200 or respuesta.status_code == 201:
    datoreserva = respuesta.json()
    print("Reserva creada con ID:", datoreserva.get("bookingid"))
    df_new = pd.DataFrame([datoreserva])
    display(df_new)
else:
    print(respuesta.text)

POST /booking - Status: 200
Reserva creada con ID: 4459


,bookingid,booking
0,4459,"{'firstname': 'Claudio', 'lastname': 'Segura',..."


# Método __PUT__ ejemplo:

In [15]:
# Usamos la reserva que acabamos de crear (si existe) para el put
#locals() devuelve un diccionario con todas las variables locales, si existe datoreserva y tiene bookingid, entonces hacemos el put
if 'datoreserva' in locals() and "bookingid" in datoreserva:
    booking_id = datoreserva["bookingid"]
    
    update_data = {
        "firstname": "Claudio Actualizado",
        "lastname": "Segura",
        "totalprice": 300000,
        "depositpaid": True,
        "bookingdates": {
            "checkin": "2026-04-16",
            "checkout": "2026-04-25"
        },
        "additionalneeds": "Habitación con vista a la montaña"
    }
    
    # Agregamos el token en los headers que pide la web para autenticar la petición
    headers = {"Cookie": f"token={token}"}
    
    respuesta = requests.put(url + f"/booking/{booking_id}", json=update_data, headers=headers)
    print("PUT /booking - Status:", respuesta.status_code)
    
    if respuesta.status_code == 200:
        df_updated = pd.DataFrame([respuesta.json()])
        display(df_updated)
        print("ID:"+str(booking_id))
    else:
        print("Error!!! MAL AHI!")
        print(respuesta.text)

PUT /booking - Status: 200


,firstname,lastname,totalprice,depositpaid,bookingdates,additionalneeds
0,Claudio Actualizado,Segura,300000,True,"{'checkin': '2026-04-16', 'checkout': '2026-04...",Habitación con vista a la montaña


ID:4459


***
## Hago Get del _id_ para probar que guardo el dato modificado con __PUT__

In [16]:
#revisamos el id
#booking_id = 2323
print("Probando con booking_id:", booking_id)

respuesta = requests.get(url + f"/booking/{booking_id}")
print("Status Code:", respuesta.status_code)

if respuesta.status_code == 200:
    datos = respuesta.json()
    df_booking = pd.DataFrame([datos])   # Convertimos el diccionario a DataFrame
    display(df_booking)
else:
    print(respuesta.text)

Probando con booking_id: 4459
Status Code: 200


,firstname,lastname,totalprice,depositpaid,bookingdates,additionalneeds
0,Claudio Actualizado,Segura,300000,True,"{'checkin': '2026-04-16', 'checkout': '2026-04...",Habitación con vista a la montaña


# Método PATCH ejemplo:

In [17]:
# la variable locals() devuelve un diccionario con todas las variables locales, si existe booking_id entonces hacemos el patch
#buscamos la variable booking_id que se creo en el put, si existe hacemos el patch, sino no hacemos nada
if 'booking_id' in locals():
    patch_data = {
        "additionalneeds": "Solo WiFi gratis",
        "totalprice": 28000
    }
    
    headers = {"Cookie": f"token={token}"}
    
    respuesta = requests.patch(url + f"/booking/{booking_id}", json=patch_data, headers=headers)
    print("PATCH /booking - Status:", respuesta.status_code)
    
    if respuesta.status_code == 200:
        df_patch = pd.DataFrame([respuesta.json()])
        display(df_patch)
    else:
        print("Error!!! MAL AHI!")
        print(respuesta.text)

PATCH /booking - Status: 200


,firstname,lastname,totalprice,depositpaid,bookingdates,additionalneeds
0,Claudio Actualizado,Segura,28000,True,"{'checkin': '2026-04-16', 'checkout': '2026-04...",Solo WiFi gratis


# Metodo DELETE ejemplo:

In [18]:
if 'booking_id' in locals():
    headers = {"Cookie": f"token={token}"}
    
    response = requests.delete(url + f"/booking/{booking_id}", headers=headers)
    print("DELETE /booking - Status:", response.status_code)
    print("Respuesta:", response.text)
    
    if response.status_code == 201 or response.status_code == 200:
        print("Reserva eliminada correctamente de booking.com")

DELETE /booking - Status: 201
Respuesta: Created
Reserva eliminada correctamente de booking.com
